# AML Transaction Monitoring & Alert Investigation Analytics
This notebook analyzes synthetic customer and transaction data for AML red flags, behavioral patterns, and customer risk scoring.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..')
customers = pd.read_csv(ROOT/'data/raw/customers.csv')
tx = pd.read_csv(ROOT/'data/processed/transactions_enriched.csv', parse_dates=['transaction_date'])
alerts = pd.read_csv(ROOT/'data/processed/alerts.csv')
risk = pd.read_csv(ROOT/'data/processed/customer_risk_features.csv')

customers.shape, tx.shape, alerts.shape, risk.shape

## 1. Transaction Overview

In [ ]:
tx[['amount']].describe()


## 2. Channel Mix

In [ ]:
tx['channel'].value_counts(normalize=True).mul(100).round(1)

## 3. High-Risk Jurisdiction Activity

In [ ]:
tx[tx['counterparty_country'].isin(['RU','IR','SY'])]\
  .groupby('customer_id')['amount'].agg(['count','sum']).sort_values('sum', ascending=False).head(20)

## 4. Structuring-Like Cash Deposits

In [ ]:
structuring = tx[(tx['channel']=='Cash Deposit') & tx['amount'].between(8000,9999.99)]
structuring.groupby('customer_id')['amount'].agg(['count','sum']).query('count >= 3').sort_values('sum', ascending=False)

## 5. Customer AML Risk Scores

In [ ]:
risk.sort_values('risk_score', ascending=False).head(25)

## 6. Alert Analysis

In [ ]:
alerts.groupby('scenario').agg(alerts=('alert_id','count'), avg_risk=('risk_score','mean'), alert_volume=('alert_amount','sum')).round(2)

## 7. EDA + Feature Engineering

This section extends the existing **AML Transaction Monitoring & Alert Investigation Analytics** work with a simple, practical EDA and feature-engineering workflow.

The transaction overview, channel mix, high-risk jurisdiction activity, structuring analysis, customer AML risk scores, and alert analysis above are kept unchanged.


### 7.1 Dataset Review & Data Quality

In [ ]:
# Review the datasets already used in the project
for name, df in [
    ('Customers', customers),
    ('Transactions', tx),
    ('Alerts', alerts),
    ('Customer Risk', risk)
]:
    print(f"\n{name}: {df.shape}")
    print("Duplicate rows:", df.duplicated().sum())
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing):
        display(missing.to_frame('missing_count'))
    else:
        print("No missing values found.")


### 7.2 Summary Statistics & Range Validation

In [ ]:
# Basic review of important AML fields
if 'amount' in tx.columns:
    display(tx[['amount']].describe().T)
    print("Negative transaction amounts:", int((tx['amount'] < 0).sum()))

if 'risk_score' in risk.columns:
    display(risk[['risk_score']].describe().T)
    print("Missing customer risk scores:", int(risk['risk_score'].isna().sum()))

if 'alert_amount' in alerts.columns:
    display(alerts[['alert_amount']].describe().T)
    print("Negative alert amounts:", int((alerts['alert_amount'] < 0).sum()))


### 7.3 Simple Outlier Review

In [ ]:
# Flag unusually large transactions for review using the IQR method.
# In AML work, outliers can be meaningful, so they are not automatically removed.
if 'amount' in tx.columns:
    q1 = tx['amount'].quantile(0.25)
    q3 = tx['amount'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    potential_outliers = tx[(tx['amount'] < lower) | (tx['amount'] > upper)]
    print("Potential transaction outliers:", len(potential_outliers))
    display(potential_outliers.head(10))


### 7.4 Feature Engineering

The features below are intentionally straightforward and tied directly to transaction-monitoring investigation.


In [ ]:
# Work on copies so the original analysis stays unchanged
tx_fe = tx.copy()
alerts_fe = alerts.copy()
risk_fe = risk.copy()

created_features = []

# High-risk jurisdiction flag based on the same countries already used above
if 'counterparty_country' in tx_fe.columns:
    tx_fe['high_risk_country_flag'] = (
        tx_fe['counterparty_country'].isin(['RU', 'IR', 'SY'])
    ).astype(int)
    created_features.append('high_risk_country_flag')

# Structuring-like cash deposit flag using the project's existing rule
if {'channel', 'amount'}.issubset(tx_fe.columns):
    tx_fe['structuring_amount_flag'] = (
        (tx_fe['channel'] == 'Cash Deposit') &
        (tx_fe['amount'].between(8000, 9999.99))
    ).astype(int)
    created_features.append('structuring_amount_flag')

# Simple transaction amount band
if 'amount' in tx_fe.columns:
    tx_fe['amount_band'] = pd.cut(
        tx_fe['amount'],
        bins=[-float('inf'), 1000, 5000, 10000, float('inf')],
        labels=['Under 1K', '1K-5K', '5K-10K', '10K+']
    )
    created_features.append('amount_band')

# Customer high-risk flag based on the top quarter of existing risk scores
if 'risk_score' in risk_fe.columns:
    threshold = risk_fe['risk_score'].quantile(0.75)
    risk_fe['high_customer_risk_flag'] = (
        risk_fe['risk_score'] >= threshold
    ).astype(int)
    created_features.append('high_customer_risk_flag')

print("Features created:", created_features)
display(tx_fe.head())
display(risk_fe.head())


### 7.5 Business Rule & KPI Validation

In [ ]:
# Recheck the core AML rules and metrics used in this project
if 'high_risk_country_flag' in tx_fe.columns:
    print("High-risk jurisdiction transactions:",
          int(tx_fe['high_risk_country_flag'].sum()))

if 'structuring_amount_flag' in tx_fe.columns:
    print("Transactions in structuring-like cash range:",
          int(tx_fe['structuring_amount_flag'].sum()))

if {'scenario', 'alert_id', 'risk_score', 'alert_amount'}.issubset(alerts_fe.columns):
    display(
        alerts_fe.groupby('scenario').agg(
            alerts=('alert_id', 'count'),
            avg_risk=('risk_score', 'mean'),
            alert_volume=('alert_amount', 'sum')
        ).round(2)
    )

if 'high_customer_risk_flag' in risk_fe.columns:
    print("Customers in higher-risk score group:",
          int(risk_fe['high_customer_risk_flag'].sum()))


### 7.6 Final Validation & Optional Export

In [ ]:
print("Final transaction dataset shape:", tx_fe.shape)
print("Final alert dataset shape:", alerts_fe.shape)
print("Final risk dataset shape:", risk_fe.shape)

print("Transaction duplicates:", tx_fe.duplicated().sum())
print("Alert duplicates:", alerts_fe.duplicated().sum())
print("Risk duplicates:", risk_fe.duplicated().sum())

# Optional enriched exports for Tableau, Streamlit, or further analysis.
# tx_fe.to_csv(ROOT/'data/processed/transactions_eda_enriched.csv', index=False)
# risk_fe.to_csv(ROOT/'data/processed/customer_aml_risk_enriched.csv', index=False)

print("EDA + Feature Engineering completed.")
